# Carga y lectura inicial de datos
Este bloque importa las librerías base y carga el archivo CSV desde Kaggle.

In [35]:
from pathlib import Path
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml import Pipeline
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("DesercionEstudiantil").getOrCreate()

local_path = Path("tbl_desercion_estudiantil_primer_anio.csv")
kaggle_path = Path("/kaggle/input/datasets/aranivadiego/tbl-desercion-estudiantil-primer-anio/tbl_desercion_estudiantil_primer_anio.csv")
csv_path = kaggle_path if kaggle_path.exists() else local_path

df = (
    spark.read.option("header", True)
    .option("sep", ";")
    .option("inferSchema", True)
    .option("nullValue", "NULL")
    .csv(str(csv_path))
)

output_path = "/kaggle/working/data/desercion_parquet"
df.write.mode("overwrite").parquet(output_path)
df = spark.read.parquet(output_path)

df.show(5)

+--------+--------+------+------------+-----------------------+---------+-----------------------+--------------------+--------------------+---------------------+-----------------+----------------+--------------------+--------------------+---------------------+-----------------+----------------+----------------------------+----------------------------+-----------------------------+--------------------+---------------------+-------+
| Carrera|IdCampus|  Sexo|CicloIngreso|InstitucionBach_Encoded|TieneBeca|PorcentajeBeca_Promedio|MateriasInscritas_C1|MateriasAprobadas_C1|MateriasReprobadas_C1|TasaAprobacion_C1|PromedioCiclo_C1|MateriasInscritas_C2|MateriasAprobadas_C2|MateriasReprobadas_C2|TasaAprobacion_C2|PromedioCiclo_C2|TotalMateriasInscritas_Anio1|TotalMateriasAprobadas_Anio1|TotalMateriasReprobadas_Anio1|TasaAprobacion_Anio1|PromedioGeneral_Anio1|Deserto|
+--------+--------+------+------------+-----------------------+---------+-----------------------+--------------------+------------

# Exploración inicial del dataset
Este bloque revisa el esquema, los nulos y la distribución de la variable objetivo con Spark.

In [36]:
print("Esquema del dataset")
df.printSchema()

print("Valores nulos por columna")
missing_counts = df.select([F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in df.columns])
missing_counts.show(truncate=False)

print("Distribución de la variable objetivo")
df.groupBy("Deserto").count().orderBy("Deserto").show()

Esquema del dataset
root
 |-- Carrera: string (nullable = true)
 |-- IdCampus: integer (nullable = true)
 |-- Sexo: integer (nullable = true)
 |-- CicloIngreso: string (nullable = true)
 |-- InstitucionBach_Encoded: double (nullable = true)
 |-- TieneBeca: integer (nullable = true)
 |-- PorcentajeBeca_Promedio: double (nullable = true)
 |-- MateriasInscritas_C1: integer (nullable = true)
 |-- MateriasAprobadas_C1: integer (nullable = true)
 |-- MateriasReprobadas_C1: integer (nullable = true)
 |-- TasaAprobacion_C1: double (nullable = true)
 |-- PromedioCiclo_C1: double (nullable = true)
 |-- MateriasInscritas_C2: integer (nullable = true)
 |-- MateriasAprobadas_C2: integer (nullable = true)
 |-- MateriasReprobadas_C2: integer (nullable = true)
 |-- TasaAprobacion_C2: double (nullable = true)
 |-- PromedioCiclo_C2: double (nullable = true)
 |-- TotalMateriasInscritas_Anio1: integer (nullable = true)
 |-- TotalMateriasAprobadas_Anio1: integer (nullable = true)
 |-- TotalMateriasReprobad

# Preparación de variables
Aquí se limpian nulos, se codifican las columnas categóricas y se arma el vector de características.

In [37]:
categorical_cols = ["Carrera", "CicloIngreso"]

df = df.fillna(0).withColumn("Deserto", F.col("Deserto").cast("int"))

indexers = [
    StringIndexer(inputCol=col, outputCol=f"{col}_idx", handleInvalid="keep")
    for col in categorical_cols
]

indexer_pipeline = Pipeline(stages=indexers)
indexer_model = indexer_pipeline.fit(df)
df_model = indexer_model.transform(df)

indexed_cols = [f"{col}_idx" for col in categorical_cols]
base_feature_cols = [
    col
    for col in df_model.columns
    if col not in ["Deserto", *categorical_cols, *indexed_cols]
 ]
feature_cols = base_feature_cols + indexed_cols

assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
data = assembler.transform(df_model).select(F.col("Deserto").alias("label"), "features")

data.show(5, truncate=False)

+-----+---------------------------------------------------------------------------------------------------------+
|label|features                                                                                                 |
+-----+---------------------------------------------------------------------------------------------------------+
|0    |[1.0,102301.0,0.23377,0.0,0.0,4.0,4.0,0.0,1.0,7.34,4.0,4.0,0.0,1.0,7.64,8.0,8.0,0.0,1.0,7.49,11.0,2.0]   |
|0    |[1.0,102302.0,0.16667,0.0,0.0,4.0,3.0,0.0,0.75,5.12,4.0,4.0,0.0,1.0,8.33,8.0,7.0,0.0,0.875,6.72,32.0,2.0]|
|0    |[1.0,102301.0,0.15873,0.0,0.0,5.0,5.0,0.0,1.0,8.33,5.0,5.0,0.0,1.0,8.68,10.0,10.0,0.0,1.0,8.51,33.0,2.0] |
|0    |[1.0,102302.0,0.05063,0.0,0.0,5.0,5.0,0.0,1.0,7.48,4.0,4.0,0.0,1.0,7.47,9.0,9.0,0.0,1.0,7.47,2.0,2.0]    |
|0    |[1.0,102302.0,0.07778,0.0,0.0,4.0,2.0,2.0,0.5,5.9,2.0,1.0,1.0,0.5,5.85,6.0,3.0,3.0,0.5,5.87,22.0,2.0]    |
+-----+---------------------------------------------------------------------------------

# División de entrenamiento y prueba
Este bloque separa los datos y deja listas las particiones para entrenar y evaluar los modelos.

In [38]:
train_df, test_df = data.randomSplit([0.7, 0.3], seed=42)

train_df.cache()
test_df.cache()

print("Cantidad de registros de entrenamiento:", train_df.count())
print("Cantidad de registros de prueba:", test_df.count())

Cantidad de registros de entrenamiento: 5380
Cantidad de registros de prueba: 2137


# Entrenamiento de modelos
Aquí se entrenan Logistic Regression y Random Forest con Spark para comparar desempeño y costo computacional.

In [39]:
import time

lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=100)
rf = RandomForestClassifier(featuresCol="features", labelCol="label", numTrees=100, seed=42, maxBins=256)

start_lr = time.perf_counter()
lr_model = lr.fit(train_df)
lr_train_time = time.perf_counter() - start_lr

start_rf = time.perf_counter()
rf_model = rf.fit(train_df)
rf_train_time = time.perf_counter() - start_rf

start_lr_pred = time.perf_counter()
lr_predictions = lr_model.transform(test_df)
lr_predictions.count()
lr_pred_time = time.perf_counter() - start_lr_pred

start_rf_pred = time.perf_counter()
rf_predictions = rf_model.transform(test_df)
rf_predictions.count()
rf_pred_time = time.perf_counter() - start_rf_pred

# Comparación final de resultados
En esta última celda se resumen métricas de desempeño y métricas de eficiencia para decidir qué modelo conviene más.

In [40]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

accuracy_evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
precision_evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedPrecision")
recall_evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="weightedRecall")
f1_evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="f1")
auc_evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")

def build_metrics(model_name, predictions, train_time, pred_time):
    return {
        "Modelo": model_name,
        "Accuracy": round(accuracy_evaluator.evaluate(predictions), 4),
        "Precision": round(precision_evaluator.evaluate(predictions), 4),
        "Recall": round(recall_evaluator.evaluate(predictions), 4),
        "F1": round(f1_evaluator.evaluate(predictions), 4),
        "ROC_AUC": round(auc_evaluator.evaluate(predictions), 4),
        "Tiempo_entrenamiento_s": round(train_time, 4),
        "Tiempo_prediccion_s": round(pred_time, 4),
    }

comparative_results = [
    build_metrics("Logistic Regression", lr_predictions, lr_train_time, lr_pred_time),
    build_metrics("Random Forest", rf_predictions, rf_train_time, rf_pred_time),
]

comparative_df = spark.createDataFrame(comparative_results)
comparative_df.show(truncate=False)

best_by_f1 = max(comparative_results, key=lambda row: (row["F1"], row["ROC_AUC"]))
fastest_train = min(comparative_results, key=lambda row: row["Tiempo_entrenamiento_s"] )
fastest_pred = min(comparative_results, key=lambda row: row["Tiempo_prediccion_s"] )

print("Metricas usadas para comparar el modelo:")
print("- Accuracy")
print("- Precision ponderada")
print("- Recall ponderado")
print("- F1")
print("- ROC AUC")
print("- Tiempo de entrenamiento")
print("- Tiempo de prediccion")
print(f"Mejor opcion segun F1 y ROC AUC: {best_by_f1['Modelo']}")
print(f"Modelo mas rapido de entrenar: {fastest_train['Modelo']}")
print(f"Modelo mas rapido de predecir: {fastest_pred['Modelo']}")

+--------+------+-------------------+---------+-------+------+----------------------+-------------------+
|Accuracy|F1    |Modelo             |Precision|ROC_AUC|Recall|Tiempo_entrenamiento_s|Tiempo_prediccion_s|
+--------+------+-------------------+---------+-------+------+----------------------+-------------------+
|0.9064  |0.89  |Logistic Regression|0.8909   |0.8301 |0.9064|3.9043                |0.0996             |
|0.9073  |0.8877|Random Forest      |0.8931   |0.8341 |0.9073|2.5616                |0.0964             |
+--------+------+-------------------+---------+-------+------+----------------------+-------------------+

Metricas usadas para comparar el modelo:
- Accuracy
- Precision ponderada
- Recall ponderado
- F1
- ROC AUC
- Tiempo de entrenamiento
- Tiempo de prediccion
Mejor opcion segun F1 y ROC AUC: Logistic Regression
Modelo mas rapido de entrenar: Random Forest
Modelo mas rapido de predecir: Random Forest


# Conclusión
En este análisis se observó que ambos modelos ofrecen un desempeño muy similar, pero Logistic Regression mostró una ligera ventaja en F1 y ROC AUC, por lo que resulta más equilibrado para la tarea de predicción de deserción estudiantil. Por su parte, Random Forest presentó un entrenamiento un poco más rápido y un rendimiento competitivo, así que también puede considerarse una alternativa válida si se prioriza eficiencia computacional. En conjunto, el uso de Spark permitió escalar el flujo de trabajo y comparar los modelos de forma reproducible dentro de Kaggle.